In [62]:
import numpy as np
import pandas as pd
from math import factorial
from scipy.stats import poisson
import math as math

df = pd.read_csv("full_wc_match_data.csv")
df = df.dropna()

print(df.isnull().sum())

match_id      0
year          0
stage         0
home_team     0
away_team     0
home_score    0
away_score    0
home_rank     0
away_rank     0
result        0
home_gdp      0
home_pop      0
away_gdp      0
away_pop      0
dtype: int64


In [63]:
def normalize(col):
    std = col.std()
    return (col - col.mean()) / std if std != 0 else col * 0
 
df["home_gdp"]  = normalize(np.log(df["home_gdp"] + 1))
df["away_gdp"]  = normalize(np.log(df["away_gdp"] + 1))
df["home_pop"]  = normalize(np.log(df["home_pop"] + 1))
df["away_pop"]  = normalize(np.log(df["away_pop"] + 1))
df["home_rank"] = normalize(-np.log(df["home_rank"]))
df["away_rank"] = normalize(-np.log(df["away_rank"]))
 
train_df = df[df["year"] < 2018].copy()
test_df  = df[df["year"] >= 2018].copy()
 
print(f"Train: {len(train_df)} matches | Test: {len(test_df)} matches")
 
def build_matrices(df):
    ones = np.ones(len(df))
    XH = np.column_stack([ones,df["home_rank"], df["home_gdp"], df["home_pop"],df["away_rank"], df["away_gdp"], df["away_pop"],ones])
    XA = np.column_stack([ones,df["away_rank"], df["away_gdp"], df["away_pop"],df["home_rank"], df["home_gdp"], df["home_pop"],np.zeros(len(df))])
    kH = df["home_score"]
    kA = df["away_score"]
    return XH, XA, kH, kA
  
def train_poisson(df, epochs=500, lr=0.0001, l2=0.01):
    XH, XA, kH, kA = build_matrices(df)
    beta = np.zeros(8)
    for epoch in range(epochs):
        lambdaH = np.clip(np.exp(XH @ beta), -10, 10)
        lambdaA = np.clip(np.exp(XA @ beta), -10, 10)
        grad = (kH - lambdaH) @ XH + (kA - lambdaA) @ XA
        grad -= l2 * beta
        grad_norm = np.linalg.norm(grad)
        if grad_norm > 1.0:
            grad /= grad_norm
 
        beta += lr * grad
 
    return beta
    
def predict_lambdas(beta, row):
    xH = np.array([1, row["home_rank"], row["home_gdp"], row["home_pop"],row["away_rank"], row["away_gdp"], row["away_pop"], 1])
    xA = np.array([1, row["away_rank"], row["away_gdp"], row["away_pop"],row["home_rank"], row["home_gdp"], row["home_pop"], 0])
    return np.exp(beta @ xH), np.exp(beta @ xA)
    
def predict_goals(lambdaH, lambdaA, max_goals=10):

    home_probs = []
    away_probs = []

    for g in range(max_goals+1):
        home_probs.append((lambdaH ** g) * math.exp(-lambdaH) / math.factorial(g))
        away_probs.append((lambdaA ** g) * math.exp(-lambdaA) / math.factorial(g))

    return home_probs, away_probs

def predict_result(lambdaH, lambdaA, max_goals=10):
    home_probs, away_probs = predict_goals(lambdaH, lambdaA, max_goals)

    P_home = 0
    P_draw = 0
    P_away = 0

    for i in range(max_goals+1):      
        for j in range(max_goals+1):  

            p = home_probs[i] * away_probs[j]

            if i > j:
                P_home += p
            elif i == j:
                P_draw += p
            else:
                P_away += p

    return P_home, P_draw, P_away

def get_actual_result(row):
    if row["home_score"] > row["away_score"]:
        return 0   
    elif row["home_score"] == row["away_score"]:
        return 1   
    else:
        return 2 
 
def compute_accuracy(df, beta):
    correct = 0
    for _, row in df.iterrows():
        lH, lA = predict_lambdas(beta, row)
        pred   = np.argmax(predict_result(lH, lA))
        if pred == get_actual_result(row):
            correct += 1
    return correct / len(df)

Train: 800 matches | Test: 128 matches


In [64]:
beta = train_poisson(train_df)
print("Beta:", beta)
 
train_acc = compute_accuracy(train_df, beta)
test_acc  = compute_accuracy(test_df, beta)
 
print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")


Beta: [ 0.03031397  0.01202242 -0.00282272  0.00305593 -0.00554948 -0.02056216
 -0.00976777  0.02949625]
Train accuracy: 0.5475
Test accuracy:  0.4922
